In [86]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm= ChatOpenAI(model="gpt-4o-mini")



In [87]:
from multiprocessing import process


@tool
def tool_name(input_param: str) -> str:
   """
   Clear description of what the tool does.
   
   Args:
       input_param (input_type): Description of this parameter
   
   Returns:
       output_type: Description of what is returned
   """
   # Function implementation
   result = process(input_param)
   return result


In [88]:
@tool
def add(a: int, b: int) -> int:
    """
    Add a and b.
    
    Args:
        a (int): first integer to be added
        b (int): second integer to be added

    Return:
        int: sum of a and b
    """
    return a + b

In [89]:
tools = [add]

llm_with_tools = llm.bind_tools(tools)

In [90]:
@tool
def subtract(a: int, b:int) -> int:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: int, b:int) -> int:
    """Multiply a and b."""
    return a * b

In [91]:
tool_map = {
    "add": add, 
    "subtract": subtract,
    "multiply": multiply
}

input_ = {
    "a": 1,
    "b": 2
}

tool_map["add"].invoke(input_)

3

In [92]:
tools = [add, subtract, multiply]

llm_with_tools = llm.bind_tools(tools)

In [93]:
query = "What is 3 + 2?"
chat_history = [HumanMessage(content=query)]

In [94]:
response_1 = llm_with_tools.invoke(chat_history)
chat_history.append(response_1)

print(type(response_1))

<class 'langchain_core.messages.ai.AIMessage'>


In [95]:
tool_calls_1 = response_1.tool_calls

tool_1_name = tool_calls_1[0]["name"]
tool_1_args = tool_calls_1[0]["args"]
tool_call_1_id = tool_calls_1[0]["id"]

print(f'tool name:\n{tool_1_name}')
print(f'tool args:\n{tool_1_args}')
print(f'tool call ID:\n{tool_call_1_id}')

tool name:
add
tool args:
{'a': 3, 'b': 2}
tool call ID:
call_Yni6NizxXE0IhtOutrB3KRL7


In [96]:
tool_response = tool_map[tool_1_name].invoke(tool_1_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_1_id)

print(tool_message)

content='5' tool_call_id='call_Yni6NizxXE0IhtOutrB3KRL7'


In [97]:
chat_history.append(tool_message)

In [98]:
answer = llm_with_tools.invoke(chat_history)
print(type(answer))
print(answer.content)

<class 'langchain_core.messages.ai.AIMessage'>
The sum of 3 and 2 is 5.


In [99]:
class ToolCallingAgent:
    def __init__(self, llm):
        self.llm_with_tools = llm.bind_tools(tools)
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        # Step 1: Initial user message
        chat_history = [HumanMessage(content=query)]

        # Step 2: LLM chooses tool
        response = self.llm_with_tools.invoke(chat_history)
        if not response.tool_calls:
            return response.contet # Direct response, no tool needed
        # Step 3: Handle first tool call
        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        # Step 4: Call tool manually
        tool_result = self.tool_map[tool_name].invoke(tool_args)

        # Step 5: Send result back to LLM
        tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
        chat_history.extend([response, tool_message])

        # Step 6: Final LLM result
        final_response = self.llm_with_tools.invoke(chat_history)
        return final_response.content

In [100]:
my_agent = ToolCallingAgent(llm)

print(my_agent.run("one plus 2"))

print(my_agent.run("one - 2"))

print(my_agent.run("three times two"))

One plus two equals three.
The result of \(1 - 2\) is \(-1\).
Three times two is 6.


In [101]:
@tool
def calculate_tip(total_bill: int, tip_percent: int) -> int:
    """Calculate tip"""
    return total_bill * tip_percent * 0.01

inputs = {
    "total_bill": 120,
    "tip_percent": 15
}
calculate_tip.invoke(inputs)


tool_map = {
    "calculate_tip": calculate_tip
}


In [105]:
class TipAgent:
    def __init__(self, llm):
        self.llm_with_tool = llm.bind_tools([calculate_tip])
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        chat_history = [HumanMessage(content=query)]
        response = llm_with_tool.invoke(chat_history)

        tool_calls = response.tool_calls
        tool_name = tool_calls[0]["name"]
        tool_args = tool_calls[0]["args"]
        tool_call_id = tool_calls[0]["id"]
        
        tool_response = tool_map[tool_name].invoke(tool_args)
        tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_id)
        
        chat_history.extend([response, tool_message])
        
        return llm_with_tool.invoke(chat_history).content

agent = TipAgent(llm)
agent.run("How much should I tip on $60 at 20%?")

'You should tip $12 on a $60 bill at 20%.'